In [1]:
import os, json, rasterio, glob
import numpy as np
from rasterio.windows import Window
from rasterio import features
from shapely.geometry import Polygon

# Configuration
RAW_DIR = '../raw_data'   
OUT_DIR = '../train_data'   
TILE_SIZE = 512
OVERLAP = 128            

def process_disaster_scenes():
    # Ensure standard output structure
    os.makedirs(os.path.join(OUT_DIR, "images"), exist_ok=True)
    os.makedirs(os.path.join(OUT_DIR, "masks"), exist_ok=True)
    
    stride = TILE_SIZE - OVERLAP 
    tif_files = glob.glob(os.path.join(RAW_DIR, "*.tif"))
    
    print(f"🔍 Found {len(tif_files)} total TIFF images (Mussett, Laura, Eruption).")

    for tif_path in tif_files:
        filename = os.path.basename(tif_path)
        # Handle various naming conventions
        file_prefix = filename.replace('.tif', '').replace('_tif', '')
        
        base_json = os.path.join(RAW_DIR, f"{file_prefix}_json.json")
        align_json = os.path.join(RAW_DIR, f"{file_prefix}_json_aligned.json")
        
        if not os.path.exists(base_json) or not os.path.exists(align_json):
            print(f"⚠️ Skipping {file_prefix}: Missing JSON alignment files.")
            continue

        print(f"🏗️ Processing Scene: {file_prefix}")

        # A. Calculate Mean Alignment Shift
        with open(align_json) as f:
            align_data = json.load(f)
        s_x = np.mean([p[1][0] - p[0][0] for p in align_data])
        s_y = np.mean([p[1][1] - p[0][1] for p in align_data])
        
        with rasterio.open(tif_path) as src:
            # B. Extract and Shift Polygons
            with open(base_json) as f:
                polys = []
                for entry in json.load(f):
                    if 'pixels' in entry:
                        # Apply shift to align buildings with the drone imagery
                        c = [(p['x'] + s_x, p['y'] + s_y) for p in entry['pixels']]
                        world_c = [src.transform * (px, py) for px, py in c]
                        polys.append(Polygon(world_c))

            # C. Systematic Overlapping Tiling
            for y in range(0, src.height - TILE_SIZE, stride):
                for x in range(0, src.width - TILE_SIZE, stride):
                    window = Window(x, y, TILE_SIZE, TILE_SIZE)
                    
                    # Rasterize the building polygons into a binary mask
                    mask = features.rasterize(
                        polys, 
                        out_shape=(TILE_SIZE, TILE_SIZE), 
                        transform=src.window_transform(window), 
                        fill=0, default_value=1, dtype=np.uint8
                    )
                    
                    # PRIORITY: Only save tiles containing building pixels
                    if np.sum(mask) > 50: 
                        tile_id = f"{file_prefix}_y{y}_x{x}.npy"
                        img_patch = src.read([1, 2, 3], window=window)
                        
                        # Save in (H, W, C) format for easier visualization later
                        np.save(os.path.join(OUT_DIR, "images", tile_id), img_patch.transpose(1, 2, 0))
                        np.save(os.path.join(OUT_DIR, "masks", tile_id), mask)

    total_images = len(os.listdir(os.path.join(OUT_DIR, "images")))
    print(f"✅ Step 1 Complete! Generated {total_images} building-focused tiles.")

# Start Preprocessing
process_disaster_scenes()

🔍 Found 10 total TIFF images (Mussett, Laura, Eruption).
🏗️ Processing Scene: MussettBayouFire-03
🏗️ Processing Scene: eruption-03
🏗️ Processing Scene: laura-01
🏗️ Processing Scene: eruption-02
🏗️ Processing Scene: MussettBayouFire-02
🏗️ Processing Scene: MussettBayouFire-01
🏗️ Processing Scene: laura-02
🏗️ Processing Scene: eruption-01
🏗️ Processing Scene: MussettBayouFire-04
🏗️ Processing Scene: MussettBayouFire-05
✅ Step 1 Complete! Generated 4515 building-focused tiles.


In [2]:
import os
import torch
import numpy as np
from torch.utils.data import Dataset, Subset, DataLoader

class DisasterFoundationDataset(Dataset):
    def __init__(self, root_dir='../train_data'):
        self.img_dir = os.path.join(root_dir, 'images')
        self.mask_dir = os.path.join(root_dir, 'masks')
        # All tiles from Fire, Hurricane, and Volcano
        self.filenames = sorted([f for f in os.listdir(self.img_dir) if f.endswith('.npy')])

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        fname = self.filenames[idx]
        img = np.load(os.path.join(self.img_dir, fname)).astype(np.float32) / 255.0
        mask = np.load(os.path.join(self.mask_dir, fname)).astype(np.float32)
        
        # Format for ResNet-50 (C, H, W)
        img_tensor = torch.from_numpy(img).permute(2, 0, 1)
        mask_tensor = torch.from_numpy(mask).unsqueeze(0)
        return img_tensor, mask_tensor

# 1. Initialize Dataset
full_dataset = DisasterFoundationDataset()

# 2. Define the Test Targets
# 1. Define only the Eruption target for the Test Set
# We are testing the model's ability to generalize to this specific volcanic scene
# 1. Define the disaster type to EXCLUDE from training
test_targets = ['eruption'] 

train_indices = []
test_indices = []

for i, fname in enumerate(full_dataset.filenames):
    # If "eruption" is in the filename, it goes to the test set
    if any(target in fname.lower() for target in test_targets):
        test_indices.append(i)
    else:
        # Everything else (Mussett and Laura) goes to training
        train_indices.append(i)

# 2. Loaders (Optimized for your MPS GPU)
train_loader = DataLoader(Subset(full_dataset, train_indices), batch_size=16, shuffle=True)
test_loader = DataLoader(Subset(full_dataset, test_indices), batch_size=1, shuffle=False)

print(f"📊 ZERO-SHOT ERUPTION SPLIT:")
print(f"🏠 Training Tiles: {len(train_indices)} (Fire + Hurricane Only)")
print(f"🔍 Testing Tiles : {len(test_indices)} (All Volcanic Eruption Scenes)")

📊 ZERO-SHOT ERUPTION SPLIT:
🏠 Training Tiles: 3284 (Fire + Hurricane Only)
🔍 Testing Tiles : 1230 (All Volcanic Eruption Scenes)


In [3]:
import os
import time
import torch
import torch.optim as optim
import segmentation_models_pytorch as smp

# --- 1. CRITICAL: Enable MPS Fallback ---
# This prevents crashes if a specific ResNet-50 math operation isn't natively 
# supported by the Mac GPU yet.
os.environ['PYTORCH_ENABLE_MPS_FALLBACK'] = '1'

# --- 2. Correct Device Selection ---
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")  # <--- This is the missing piece for your Mac!
else:
    device = torch.device("cpu")

print(f"🚀 Hardware Confirmed! Training on: {device.type.upper()}")

# --- 3. Architecture (ResNet-50) ---
model = smp.Unet(
    encoder_name="resnet50", 
    encoder_weights="imagenet", 
    in_channels=3, 
    classes=1, 
    activation='sigmoid'
).to(device)

# --- 4. Building-Priority Loss & Optimization ---
criterion = smp.losses.TverskyLoss(mode='binary', alpha=0.8, beta=0.2)
optimizer = optim.AdamW(model.parameters(), lr=0.0001, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingWarmRestarts(optimizer, T_0=5)

# --- 5. Optimized Training Loop ---
best_val_loss = float('inf')

print(f"🔥 Starting Training on {len(train_loader.dataset)} tiles...")

for epoch in range(30):
    start_time = time.time() # Track speed
    
    # --- TRAINING ---
    model.train()
    epoch_train_loss = 0
    for imgs, masks in train_loader:
        # Pushing data to Apple Silicon Unified Memory
        imgs, masks = imgs.to(device), masks.to(device)
        
        optimizer.zero_grad()
        loss = criterion(model(imgs), masks)
        loss.backward()
        optimizer.step()
        epoch_train_loss += loss.item()
    
    scheduler.step()
    
    # --- VALIDATION ---
    model.eval()
    epoch_test_loss = 0
    with torch.no_grad():
        for imgs, masks in test_loader:
            imgs, masks = imgs.to(device), masks.to(device)
            epoch_test_loss += criterion(model(imgs), masks).item()
    
    avg_test = epoch_test_loss / len(test_loader)
    epoch_duration = time.time() - start_time
    
    # --- SAVE BEST MODEL ---
    if avg_test < best_val_loss:
        best_val_loss = avg_test
        torch.save(model.state_dict(), "foundation_multi_disaster.pth")
        status = "🌟 Best Saved!"
    else:
        status = ""
        
    print(f"Epoch [{epoch+1}/30] | Train: {epoch_train_loss/len(train_loader):.4f} | "
          f"Test: {avg_test:.4f} | Time: {epoch_duration:.1f}s {status}")

print(f"✅ Training Complete. Global Best Test Loss: {best_val_loss:.4f}")

🚀 Hardware Confirmed! Training on: MPS
🔥 Starting Training on 3284 tiles...
Epoch [1/30] | Train: 0.6798 | Test: 0.8809 | Time: 217.5s 🌟 Best Saved!
Epoch [2/30] | Train: 0.6689 | Test: 0.8812 | Time: 216.0s 
Epoch [3/30] | Train: 0.6581 | Test: 0.8830 | Time: 215.9s 
Epoch [4/30] | Train: 0.6572 | Test: 0.8851 | Time: 216.2s 
Epoch [5/30] | Train: 0.6575 | Test: 0.8860 | Time: 215.5s 
Epoch [6/30] | Train: 0.6684 | Test: 0.8815 | Time: 211.5s 
Epoch [7/30] | Train: 0.6581 | Test: 0.8835 | Time: 203.7s 
Epoch [8/30] | Train: 0.6483 | Test: 0.8852 | Time: 215.2s 
Epoch [9/30] | Train: 0.6518 | Test: 0.8848 | Time: 215.0s 
Epoch [10/30] | Train: 0.6480 | Test: 0.8867 | Time: 215.9s 
Epoch [11/30] | Train: 0.6496 | Test: 0.8869 | Time: 215.1s 
Epoch [12/30] | Train: 0.6543 | Test: 0.8836 | Time: 215.3s 
Epoch [13/30] | Train: 0.6504 | Test: 0.8864 | Time: 215.0s 
Epoch [14/30] | Train: 0.6470 | Test: 0.8874 | Time: 203.3s 
Epoch [15/30] | Train: 0.6487 | Test: 0.8877 | Time: 216.2s 
Epoch

In [4]:
import torch
import numpy as np
import os
import segmentation_models_pytorch as smp

def evaluate_foundation_metrics(loader, model_path):
    # --- 1. Setup Device (Correctly Indented) ---
    if torch.cuda.is_available():
        device = torch.device("cuda")
    elif torch.backends.mps.is_available():
        device = torch.device("mps")
    else:
        device = torch.device("cpu")

    print(f"🚀 Evaluating on: {device.type.upper()}")
    
    # --- 2. Initialize ResNet-50 Architecture ---
    model = smp.Unet(
        encoder_name="resnet50", 
        in_channels=3, 
        classes=1, 
        activation='sigmoid'
    ).to(device)
    
    # --- 3. Load the Weights ---
    if not os.path.exists(model_path):
        print(f"❌ Error: {model_path} not found.")
        return
        
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    # --- 4. Pixel Counters ---
    tp, fp, tn, fn = 0, 0, 0, 0
    print(f"🔬 Analyzing Foundation Test Set (Mussett, Laura, Eruption)...")
    
    with torch.no_grad():
        for imgs, masks in loader:
            imgs, masks = imgs.to(device), masks.to(device).float()
            
            outputs = model(imgs)
            preds = (outputs > 0.5).float()
            
            p = preds.view(-1)
            m = masks.view(-1)
            
            tp += (p * m).sum().item()
            fp += (p * (1 - m)).sum().item()
            tn += ((1 - p) * (1 - m)).sum().item()
            fn += ((1 - p) * m).sum().item()

    # --- 5. Metrics Calculation ---
    epsilon = 1e-7
    precision = tp / (tp + fp + epsilon)
    recall = tp / (tp + fn + epsilon)
    f1 = 2 * (precision * recall) / (precision + recall + epsilon)
    iou = tp / (tp + fp + fn + epsilon)
    specificity = tn / (tn + fp + epsilon)
    accuracy = (tp + tn) / (tp + tn + fp + fn + epsilon)
    balanced_acc = (recall + specificity) / 2

    print(f"\n--- 🌍 GLOBAL DISASTER FOUNDATION RESULTS ---")
    print(f"IoU              : {iou:.4f}")
    print(f"Dice             : {f1:.4f}")
    print(f"Precision        : {precision:.4f}")
    print(f"Recall (Target)  : {recall:.4f}")
    print(f"F1-score         : {f1:.4f}")
    print(f"Specificity      : {specificity:.4f}")
    print(f"Balanced Acc.    : {balanced_acc:.4f}")
    print(f"Overall Accuracy : {accuracy:.4f}")

# --- Execute ---
# Note: Ensure test_loader exists in your current session!
evaluate_foundation_metrics(test_loader, "foundation_multi_disaster.pth")

🚀 Evaluating on: MPS
🔬 Analyzing Foundation Test Set (Mussett, Laura, Eruption)...

--- 🌍 GLOBAL DISASTER FOUNDATION RESULTS ---
IoU              : 0.2270
Dice             : 0.3701
Precision        : 0.2439
Recall (Target)  : 0.7666
F1-score         : 0.3701
Specificity      : 0.7710
Balanced Acc.    : 0.7688
Overall Accuracy : 0.7706
